# Complexity-06 — Simuler un circuit quantique classiquement

**Navigation** : [<< Complexity-05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) | [Index de la série](README.md) | [Version Python](Complexity-06-Simuler-Circuit-Quantique-Classiquement-Python.ipynb) | [Complexity-06b — Aaronson-Gottesman, stabilisation et déquantification >>](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb)

> **Public : Licence.** Le pont avec la déquantification et la stabilisation de Gottesman est
> approfondi dans le carnet [06b](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb) ;
> celui-ci compare deux représentations d’une distribution uniforme connue. Il ne mesure pas une borne inférieure générale de simulation.

## Ce que ce notebook suppose

- [05 — Compter est plus dur que vérifier : #P et la permanente](Complexity-05-CountingHarder-Permanent-Python.ipynb) :
  la classe **#P**, la distinction entre **décider** et **compter**, et la formule de Ryser ;
- [05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) :
  le **boson sampling**, dont les probabilités sont liées à des permanentes : échantillonner ne constitue pas un oracle de comptage exact ;
- manipuler une matrice de Pauli et un produit tensoriel.

## Ce que vous saurez faire à la fin

1. énoncer ce que signifie **simuler classiquement** un circuit quantique (sortie
   probabiliste sur n qubits = 2^n amplitudes complexes) ;
2. donner un **invariant de Pauli** sous tension : un circuit Clifford (H, S, CNOT) sur n
   qubits se décrit par un tableau de taille **O(n²)** et non 2^n amplitudes — algorithme de Gottesman-Knill ;
3. **mesurer** le coût d’un vecteur explicite et d’un tirage direct sur la même loi uniforme, sans les confondre avec une séparation de classes ; rappeler le statut
   **BQP ⊆ PP ⊆ P#P** (theoreme ; l’ouverte est BQP vs P / PH) ;
4. distinguer deux régimes de coût pour la simulation classique : **échantillonner** vs
   **calculer la distribution de probabilité exacte** (Aaronson-Arkhipov 2011 pour le boson sampling ; folklore pour les circuits universels) ;
5. énoncer ce que la **simulation classique** d'un circuit quantique universel nous
   apprend sur la frontière BQP/P.

**Durée estimée** : 45 minutes.

**Environnement C#** : .NET 9+, noyau .NET Interactive ; System.Numerics.Complex et BCL. Aucun service ni QPU. Les PRNG NumPy et System.Random diffèrent malgré la même graine 20261007.


## 1. Ce que simuler classiquement signifie

Un circuit quantique à k portes et n qubits prépare un état

> |ψ⟩ = U_k · U_{k-1} · ... · U_1 · |0^n⟩

où chaque U_i est une porte (Hadamard, phase, CNOT, Toffoli…). L'état est un vecteur de
2^n amplitudes complexes ; la mesure du registre de n qubits renvoie une loi categorielle sur les 2^n chaînes de bits possibles
(la mesure d'un qubit seul projete sur la base computationnelle donne bien une Bernoulli à 2 issues,
mais c'est la mesure du registre qui est sur 2^n issues). **Simuler classiquement** le circuit, c'est
produire des échantillons de cette distribution — ou, plus exigeant, calculer exactement
les 2^n probabilités.

La question de la **simulation polynomiale** d'un circuit quantique arbitraire est l'une
des grandes questions ouvertes de la théorie : un algorithme déterministe polynomial décidant tous les langages BQP donnerait P = BQP. Un simulateur classique probabiliste polynomial reproduisant fidèlement leurs sorties, avec une précision préservant une erreur bornée, donnerait BPP = BQP ; il ne fournit pas à lui seul une décision sans hasard. Ici BPP désigne le temps polynomial probabiliste à erreur bornée, et P ⊆ BPP ⊆ BQP. Le coût d’un simulateur particulier ne démontre aucune séparation.

Ce notebook explore une **fenêtre** dans cette question : la famille **Clifford** (Hadamard,
phase, CNOT) — où la simulation classique redevient polynomiale par l'algorithme de
Gottesman-Knill (1998). En dehors de Clifford, on entre dans la zone où la simulation
naïve coûte O(2^n).

In [1]:
#nullable enable
using System;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Numerics;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
int[] Tirer(double[] probabilites, int nombre, Random rng)
{
    if(nombre<0 || probabilites.Length==0 || probabilites.Any(p=>!double.IsFinite(p)||p<0)) throw new ArgumentException("Distribution invalide.");
    double masse=probabilites.Sum();
    if(Math.Abs(masse-1)>1e-9) throw new ArgumentException("La masse doit valoir 1.");
    var cdf=new double[probabilites.Length]; double cumul=0;
    for(int i=0;i<cdf.Length;i++){cumul+=probabilites[i];cdf[i]=cumul;}
    cdf[^1]=1;
    var echantillons=new int[nombre];
    for(int j=0;j<nombre;j++){double u=rng.NextDouble();int lo=0,hi=cdf.Length-1;while(lo<hi){int m=lo+(hi-lo)/2;if(u<cdf[m])hi=m;else lo=m+1;}echantillons[j]=lo;}
    return echantillons;
}
Console.WriteLine($".NET {Environment.Version}; System.Numerics.Complex; seed 20261007; sampler CDF binaire prêt.");

.NET 10.0.14; System.Numerics.Complex; seed 20261007; sampler CDF binaire prêt.


## 2. La barrière exponentielle : simuler naïvement

Un circuit à n qubits a 2^n amplitudes. Pour un circuit universel arbitraire (Hadamard +
T + CNOT par exemple), la méthode qui stocke toutes les amplitudes demande O(2^n) mémoire et un travail exponentiel pour les parcourir. Ce coût décrit cette représentation explicite, sans être une borne inférieure générale pour échantillonner. Pour la **mesure** sur un registre de n qubits, l'algorithme le plus naïf accumule
toutes les amplitudes au carré ; cette représentation devient coûteuse vers n ≈ 30, y compris pour une distribution dont une autre représentation serait simple.

Mesurons cette croissance : combien de temps demande une simulation vectorielle pour un
circuit de Hadamard uniquement (préparation d'une superposition uniforme) à n = 4, 8, 16,
20 qubits ?

In [2]:
#nullable enable
(double Millisecondes,int[] Indices,double Masse,int Dimension) SimulerHadamardNaive(int n,int nombre=4096)
{
    int dim=1<<n; var rng=new Random(20261007);
    _=Tirer(Enumerable.Repeat(1.0/dim,dim).ToArray(),64,rng); // chauffe isolée, même dimension
    var chrono=Stopwatch.StartNew();
    var etat=new Complex[dim]; double amplitude=1/Math.Sqrt(dim);
    Array.Fill(etat,new Complex(amplitude,0));
    var probabilites=new double[dim];
    for(int i=0;i<dim;i++) probabilites[i]=etat[i].Real*etat[i].Real+etat[i].Imaginary*etat[i].Imaginary;
    var indices=Tirer(probabilites,nombre,rng);
    chrono.Stop();
    if(indices.Length!=nombre || indices.Any(i=>i<0||i>=dim))throw new Exception("Tirages hors registre.");
    return(chrono.Elapsed.TotalMilliseconds,indices,probabilites.Sum(),dim);
}
foreach(int n in new[]{4,8,16,20})
{
    var r=SimulerHadamardNaive(n);
    Console.WriteLine($"n={n,2}, dim={r.Dimension,7} : {r.Millisecondes,9:F3} ms; 4096 tirages; masse={r.Masse:F12}; {r.Indices.Distinct().Count()} issues observées.");
}
Console.WriteLine("Périmètre mesuré : allocation/état, probabilités, CDF et 4096 tirages ; chauffe exclue.");

n= 4, dim=     16 :     0.950 ms; 4096 tirages; masse=1.000000000000; 16 issues observées.


n= 8, dim=    256 :     0.413 ms; 4096 tirages; masse=1.000000000000; 256 issues observées.


n=16, dim=  65536 :     8.385 ms; 4096 tirages; masse=1.000000000000; 3987 issues observées.


n=20, dim=1048576 :   161.432 ms; 4096 tirages; masse=1.000000000000; 4092 issues observées.


Périmètre mesuré : allocation/état, probabilités, CDF et 4096 tirages ; chauffe exclue.


Le coût mémoire du vecteur d'état croît comme 2^n nombres complexes (16 octets par
amplitude sur float64). Au-delà de n = 30 environ, la mémoire d'un poste de travail
standard (16 Go) déborde (2^30 × 16 o = 16 GiB, soit environ 17,2 Go décimaux, pour le vecteur d'état
seul ; les probas float64 et la marge système portent la limite pratique
vers n ≈ 28) ; au-delà de n = 40, il faut une machine HPC.

Cette barrière de représentation motive la recherche de descriptions plus compactes :
pour n suffisamment grand, le vecteur explicite dépasse les ressources disponibles. Cela ne démontre pas un avantage quantique : la loi uniforme du banc dispose précisément d’un tirage classique direct efficace. L'enjeu
théorique est : **quelle est la classe** des circuits qu'on peut simuler en **polynomial**
classique ?
Les chronos affichés incluent construction, probabilités et CDF : ils varient avec le JIT, les allocations et le matériel. Les petites tailles ne sont pas nécessairement ordonnées en temps ; la taille du vecteur et la masse contrôlée décrivent précisément le travail exécuté.


### Exercice 1 — Décompte #P de la simulation universelle

On a vu qu'un circuit quantique produit une distribution de mesure sur 2^n chaînes. Pour
un circuit universel arbitraire, **calculer exactement cette distribution** est #P-dur
en général (le carnet [05b](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb)
le démontre pour le boson sampling via la permanente).

**Travail** :
1. Choisir un circuit jouet de 4 qubits utilisant H, T et CNOT ;
2. Construire la matrice unitaire complète (taille 16 × 16) ;
3. Calculer les 16 probabilités en sortie (vecteur d'état × mesure) ;
4. Vérifier que la somme des probabilités vaut 1, et observer la **distribution**
   (est-elle proche de l'uniforme, piquée sur quelques chaînes, autre ?) ;
5. Comparer au temps de calcul de la simulation Gottesman-Knill sur ce même circuit
   (cas dégénéré : pas de porte T → la simulation polynomiale doit suffire).

Le tableau stabilisateur ne traite pas directement T. Cela ne prouve ni qu’un T impose un coût exponentiel, ni que le circuit particulier est universel.


**API C#** : fournir un CircuitReponse contenant la liste des portes (H/T/CX, qubits 0 à 3), la matrice 16×16 et les seize probabilités. La garde confronte toutes les colonnes au plan de portes et les probabilités à la colonne initiale ; elle n’accepte pas une matrice normalisée arbitraire. Le jeu de portes est universel, sans que votre circuit fini le soit à lui seul.

Convention : qubit 0 est le bit de poids faible ; les portes s’appliquent dans l’ordre de la liste, par multiplication à gauche.


In [3]:
#nullable enable
record Porte(string Nom,int Qubit,int Cible=-1);
record CircuitReponse(Porte[] Portes,Complex[,] Matrice,double[] Probabilites);
Complex[] AppliquerPortes(Complex[] entree,Porte[] portes)
{
    var v=(Complex[])entree.Clone();
    foreach(var p in portes)
    {
        if(p.Qubit<0||p.Qubit>=4 || !new[]{"H","T","CX"}.Contains(p.Nom))throw new Exception("Porte/qubit invalide.");
        int bit=1<<p.Qubit;
        if(p.Nom=="H")for(int i=0;i<16;i++)if((i&bit)==0){var a=v[i];var b=v[i|bit];v[i]=(a+b)/Math.Sqrt(2);v[i|bit]=(a-b)/Math.Sqrt(2);}
        if(p.Nom=="T")for(int i=0;i<16;i++)if((i&bit)!=0)v[i]*=Complex.FromPolarCoordinates(1,Math.PI/4);
        if(p.Nom=="CX")
        {
            if(p.Cible<0||p.Cible>=4||p.Cible==p.Qubit)throw new Exception("CNOT invalide.");
            int cible=1<<p.Cible;
            for(int i=0;i<16;i++)if((i&bit)!=0&&(i&cible)==0)(v[i],v[i|cible])=(v[i|cible],v[i]);
        }
    }
    return v;
}
void VerifierCircuit(CircuitReponse reponse)
{
    if(reponse.Portes is null||reponse.Matrice is null||reponse.Probabilites is null||reponse.Portes.Length==0||reponse.Matrice.GetLength(0)!=16||reponse.Matrice.GetLength(1)!=16||reponse.Probabilites.Length!=16)throw new Exception("Dimensions du circuit incorrectes.");
    if(!new[]{"H","T","CX"}.All(n=>reponse.Portes.Any(p=>p.Nom==n)))throw new Exception("Utiliser H, T et CNOT dans le plan.");
    for(int colonne=0;colonne<16;colonne++)
    {
        var entree=new Complex[16];entree[colonne]=Complex.One;
        var reference=AppliquerPortes(entree,reponse.Portes);
        for(int ligne=0;ligne<16;ligne++){var z=reponse.Matrice[ligne,colonne];if(!double.IsFinite(z.Real)||!double.IsFinite(z.Imaginary)||Complex.Abs(z-reference[ligne])>1e-10)throw new Exception("Matrice incompatible avec les portes.");}
    }
    for(int i=0;i<16;i++){double p=reponse.Probabilites[i];double attendu=reponse.Matrice[i,0].Magnitude*reponse.Matrice[i,0].Magnitude;if(!double.IsFinite(p)||p<0||Math.Abs(p-attendu)>1e-10)throw new Exception("Probabilités incompatibles avec la matrice.");}
    if(Math.Abs(reponse.Probabilites.Sum()-1)>1e-10)throw new Exception("Masse non normalisée.");
}
CircuitReponse? circuitEtudiant=null; // TODO étudiant : portes, matrice et probabilités
if(circuitEtudiant is null)Console.WriteLine("Exercice 1 à compléter : circuit à 4 qubits, matrice 16×16 et seize probabilités.");
else{VerifierCircuit(circuitEtudiant);Console.WriteLine("Matrice confrontée aux seize états de base et distribution normalisée vérifiées.");}

Exercice 1 à compléter : circuit à 4 qubits, matrice 16×16 et seize probabilités.


## 3. Le résultat de Gottesman-Knill : une description polynomiale

Une famille de portes joue un rôle particulier en informatique quantique : les portes
**Clifford** (Hadamard, phase S, CNOT). Aaronson & Gottesman (2004) ont montré qu'un
circuit construit uniquement avec ces portes se simule classiquement en **polynomial**,
avec un tableau d’état de taille **O(n²)** ; le coût total dépend aussi des portes et des mesures effectuées. L'idée : l'état d'un circuit Clifford
peut être décrit non pas par ses 2^n amplitudes, mais par une **matrice de Pauli** sur
n qubits — un objet de taille O(n²) sur {I, X, Y, Z}^n.

Ce théorème, ce carnet ne le démontre pas : la vraie simulation stabilisatrice — table
de Pauli mise à jour porte par porte — est portée par le carnet jumeau
[06b — Aaronson-Gottesman](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb).
La cellule suivante fait un geste plus modeste, et c'est le geste honnête à cette échelle :
sur le cas concret d'un circuit Clifford à n qubits (Hadamard puis CNOT) dont la
distribution finale de mesure est **uniforme** (à un rephasage près), elle chronomètre un
**échantillonnage direct** — tirer chaque bit selon une distribution connue à l'avance,
en O(n) par échantillon, sans construire le circuit ni maintenir de table de
stabilisateurs. Le banc matérialise la *conséquence* du théorème (quand la structure est
connue, l'échantillon est bon marché), pas sa preuve.

In [4]:
#nullable enable
(double Millisecondes,byte[,] Bits) EchantillonnageDirect(int n,int nombre=4096)
{
    var rng=new Random(20261007);
    for(int i=0;i<64*n;i++) _=rng.Next(2);
    var chrono=Stopwatch.StartNew();
    var bits=new byte[nombre,n];
    for(int i=0;i<nombre;i++)for(int q=0;q<n;q++)bits[i,q]=(byte)rng.Next(2);
    chrono.Stop();
    return(chrono.Elapsed.TotalMilliseconds,bits);
}
foreach(int n in new[]{4,8,16,20,24,28})
{
    var r=EchantillonnageDirect(n);int un=0;
    foreach(byte b in r.Bits){if(b>1)throw new Exception("Bit invalide.");un+=b;}
    Console.WriteLine($"n={n,2} : {r.Millisecondes,8:F3} ms; {r.Bits.GetLength(0)} échantillons de {r.Bits.GetLength(1)} bits; fraction de 1={(double)un/r.Bits.Length:F4}.");
}
Console.WriteLine("Tirage direct de |+>^n : loi connue, pas de tableau stabilisateur et pas de circuit générique simulé.");

n= 4 :    3.545 ms; 4096 échantillons de 4 bits; fraction de 1=0.4893.


n= 8 :    0.357 ms; 4096 échantillons de 8 bits; fraction de 1=0.4925.


n=16 :    1.018 ms; 4096 échantillons de 16 bits; fraction de 1=0.4980.


n=20 :    0.832 ms; 4096 échantillons de 20 bits; fraction de 1=0.4988.


n=24 :    2.591 ms; 4096 échantillons de 24 bits; fraction de 1=0.5000.


n=28 :    2.223 ms; 4096 échantillons de 28 bits; fraction de 1=0.5003.


Tirage direct de |+>^n : loi connue, pas de tableau stabilisateur et pas de circuit générique simulé.


Le coût par bit est constant dans cette implémentation directe, et le coût total est O(n) par échantillon : aucune
construction de 2^n amplitudes, aucune matrice 2^n × 2^n. Cette loi est uniforme et connue analytiquement : le tirage direct est exact dans ce cas particulier. Ce banc ne caractérise pas les circuits corrélés généraux ni la difficulté du boson sampling. Le coût total du tirage direct croît avec le nombre de bits produits, O(n) par échantillon.

### Exercice 2 — Distinction échantillonner / calculer (Aaronson-Arkhipov)

**Travail** :
1. Lire (ou résumer) le résultat Aaronson-Arkhipov (2011) : la distinction
   **échantillonner vs calculer** pour le boson sampling est le paradigme canonique
   de la frontière calculable / échantillonnable classiquement ; pour un circuit
   quantique UNIVERSEL sur n qubits, **calculer la distribution exacte** est
   généralement #P-dur (cf carnet 05b), tandis qu'**échantillonner** peut rester
   faisable en O(n) par échantillon si la structure du circuit le permet ;
2. Échantillonner depuis la distribution exacte calculée à l'exercice 1 : après le coût
   de **construction** (matrice 16 × 16 et distribution calculées une fois pour toutes),
   mesurer le coût de **tirage** — temps pour 1000 échantillons tirés dans cette
   distribution — et le comparer à l'échantillonnage direct du §3 sur Clifford
   (O(n) par échantillon, aucun coût de construction) ;
3. Conjecturer : pour quel type de problème (échantillonner vs calculer la distribution)
   la simulation classique est-elle acceptable en pratique ?


**API C#** : fournir tirageEtudiant, qui reçoit probabilités, nombre et Random et renvoie les indices tirés. Les deux distributions ponctuelles et deux lois pondérées non dégénérées servent de contre-exemples aux réponses constantes et aux tirages ignorant les poids. Pour les lois pondérées, le diagnostic utilise 4096 tirages avec graines 20261007 et 11, et un écart empirique maximal de 0,06 par catégorie. Ce diagnostic fini ne prouve pas une loi générale. La garde mesure elle-même les 1000 tirages ; elle ne lit aucune durée déclarée. L’exercice 1 doit d’abord fournir sa distribution.


In [5]:
#nullable enable
Func<double[],int,Random,int[]>? tirageEtudiant=null; // TODO étudiant : tirages depuis la distribution
if(tirageEtudiant is not { } sampler || circuitEtudiant is not { } circuit)Console.WriteLine("Exercice 2 à compléter : distribution de l'exercice 1, puis fonction de tirage.");
else
{
    foreach(var p in new[]{new[]{1d,0d,0d,0d},new[]{0d,0d,0d,1d}})
    {
        var test=sampler(p,32,new Random(7));int attendu=p[0]==1?0:3;
        if(test.Length!=32 || test.Any(i=>i!=attendu))throw new Exception("Sampler incorrect sur une loi ponctuelle.");
    }
    foreach(var (p,seed) in new[]{(new[]{.25,.75},20261007),(new[]{0d,.1,.2,.7},11)})
    {
        var test=sampler(p,4096,new Random(seed));
        if(test.Length!=4096 || test.Any(i=>i<0||i>=p.Length))throw new Exception("Échantillon de diagnostic invalide.");
        var comptes=new int[p.Length];foreach(int index in test)comptes[index]++;
        for(int j=0;j<p.Length;j++)if((p[j]==0&&comptes[j]!=0)||Math.Abs(comptes[j]/4096d-p[j])>.06)throw new Exception("Diagnostic pondéré échoué : les fréquences ne suivent pas les poids.");
    }
    Console.WriteLine("Lois ponctuelles et deux diagnostics pondérés vérifiés ; contrôle fini, pas preuve générale du sampler.");
    var chrono=Stopwatch.StartNew();
    var indices=sampler(circuit.Probabilites,1000,new Random(20261007));
    chrono.Stop();
    if(indices.Length!=1000 || indices.Any(i=>i<0||i>=16||circuit.Probabilites[i]<=0))throw new Exception("Indices ou support incompatibles avec la distribution.");
    var direct=EchantillonnageDirect(4,1000);
    Console.WriteLine($"1000 tirages depuis distribution préconstruite : {chrono.Elapsed.TotalMilliseconds:F3} ms ; direct uniforme : {direct.Millisecondes:F3} ms. Construction exclue de ces deux chronos.");
    Console.WriteLine("Les lois comparées peuvent différer ; ce contrôle ne prouve pas une équivalence statistique générale.");
}

Exercice 2 à compléter : distribution de l'exercice 1, puis fonction de tirage.


## 4. Au-delà de Clifford : la porte T et la question BQP/P

Le jeu de portes Clifford complété par T = diag(1, exp(iπ/4)) est universel, avec une précision d’approximation donnée. Un circuit particulier contenant un T ne devient pas à lui seul universel ni nécessairement difficile à simuler. La question de la simulation polynomiale d'un
tel circuit est précisément la **frontière BQP/P** :

    BQP ⊆ P   ?   (ouvert)

où **BQP** (Bounded-Error Quantum Polynomial time) est la classe des problèmes qu'un
ordinateur quantique résout en temps polynomial avec probabilité d'erreur bornée. Comme P ⊆ BQP, une inclusion BQP ⊆ P donnerait l’égalité pour les problèmes de décision. Si cette inclusion est fausse,
il existe des problèmes que le quantique résout et le classique ne sait pas résoudre en
polynomial.

La simulation vectorielle générale d’un circuit à n qubits et profondeur p coûte **O(2^n · poly(n, p))** — sans avancée algorithmique, cette
barrière reste exponentielle. C'est cette barrière qui motive la promesse algorithmique
du quantique ; c'est aussi elle qui rend la simulation classique **utilisable** sur les
cas Clifford, et rapidement coûteuse pour de grands circuits sans structure exploitable ; de petits circuits restent simulables.
La nature de l’algorithme classique compte : décision déterministe polynomiale pour tout BQP implique P = BQP ; simulation probabiliste fidèle polynomiale implique BPP = BQP. Ces deux conclusions ne sont pas interchangeables, et ce banc ne démontre aucune des deux.


### Exercice 3 — La frontière BQP/P

**Travail** :
1. Énoncer (en deux phrases) la définition de BQP ;
2. Rappeler (en une phrase) ce que la barrière O(2^n) classique nous dit sur
   la séparation BQP vs P / PH (rappel : BQP ⊆ PP ⊆ P^#P est un théorème ;
   **l'ouverte** est BQP vs P / PH, pas l'inclusion BQP ⊂ P^#P) ;
3. Conjecturer (en trois phrases maximum) : si demain un algorithme classique en
   **O(poly(n))** pour les circuits universels était trouvé, que deviendrait la valeur
   algorithmique du calcul quantique ?
**Contrôle** : renseigner les cinq repères logiques du dictionnaire ci-dessous, puis rédiger la définition de BQP et la conjecture. Le dictionnaire est vérifié ; le texte reste un argument à discuter, sans certification automatique.

Distinguez dans votre raisonnement décision déterministe et simulation probabiliste : la première hypothèse concerne P, la seconde BPP. Définir BQP inclut un temps polynomial et une erreur bornée sur les deux types d’instances, amplifiable par répétition.


In [6]:
#nullable enable
Dictionary<string,bool>? reperesBqp=null; // TODO étudiant : cinq clés décrites ci-dessous
string? definitionBqp=null, conjectureBqp=null; // TODO étudiant : définition et conjecture argumentées
var clesBqp=new[]{"p_inclus_bqp","decision_deterministe_bqp_implique_p_egal_bqp","simulation_probabiliste_fidele_bqp_implique_bpp_egal_bqp","un_t_prouve_cout_exponentiel","table_temps_prouve_separation"};
Console.WriteLine("Clés attendues : "+string.Join(", ",clesBqp));
if(reperesBqp is null)Console.WriteLine("Exercice 3 à compléter : repères, définition BQP et conjecture.");
else
{
    if(!reperesBqp.Keys.ToHashSet().SetEquals(clesBqp) || !reperesBqp[clesBqp[0]] || !reperesBqp[clesBqp[1]] || !reperesBqp[clesBqp[2]] || reperesBqp[clesBqp[3]] || reperesBqp[clesBqp[4]])throw new Exception("Repères BQP erronés : séparer inclusions connues et conclusions non démontrées.");
    Console.WriteLine("Cinq repères logiques vérifiés ; aucune séparation BQP/P déduite des chronos.");
    Console.WriteLine(string.IsNullOrWhiteSpace(definitionBqp)||string.IsNullOrWhiteSpace(conjectureBqp)?"Texte à compléter : erreur bornée, temps polynomial, et conjecture.":"Texte fourni : à lire et discuter ; sa justesse n'est pas certifiée par la garde.");
}

Clés attendues : p_inclus_bqp, decision_deterministe_bqp_implique_p_egal_bqp, simulation_probabiliste_fidele_bqp_implique_bpp_egal_bqp, un_t_prouve_cout_exponentiel, table_temps_prouve_separation


Exercice 3 à compléter : repères, définition BQP et conjecture.


## Exercices — bilan

1. Après la simulation vectorielle : construire le circuit à 4 qubits et vérifier sa matrice 16×16 et sa distribution.
2. Après le tirage direct : mesurer 1000 tirages de la distribution construite, hors coût de construction, et comparer les périmètres.
3. Après les limites de simulation : définir BQP, distinguer les inclusions connues d’une conjecture et rédiger l’argumentation.

Les trois stubs restent ouverts ; aucune réponse étudiante n’est fournie.


## À retenir

Ce carnet a posé la frontière de la simulation classique d'un circuit quantique par le
biais de deux résultats canoniques :

- le coût exponentiel O(2^n) de la représentation vectorielle explicite, qui sépare la
  zone où la simulation classique naïve reste praticable (n petit) de celle où elle
  devient intraitable (n ≥ 30) ;
- l'**algorithme de Gottesman-Knill** (1998), qui montre que la sous-famille **Clifford**
  (H, S, CNOT) admet une description d’état O(n²) et une simulation polynomiale, donc compatible avec une
  simulation classique efficace des portes et mesures stabilisatrices, sans matérialiser toutes les probabilités.
Le pont avec la **déquantification Aaronson-Gottesman** et la **stabilisation** est
approfondi dans le carnet [06b](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb).

**Sources** :
- Aaronson & Arkhipov, *The Computational Complexity of Linear Optics*, Theory of Computing 9 (2013), pp. 143-252 (boson sampling : distinction echantillonner / calculer) ;
- Aaronson & Gottesman, *Improved Simulation of Stabilizer Circuits*, Phys. Rev. A 70 (2004) ;
- Nielsen & Chuang, *Quantum Computation and Quantum Information*, 10th anniv. ed. (2010),
  ch. 10 sur la stabilité et la simulation ;
- Aaronson, *BQP and the Polynomial Hierarchy* (2009), pour la conjecture BQP ⊄ PH.

## Vérifiez votre compréhension

1. Vers $n \approx 28$–$30$, le vecteur d'état déborde la mémoire d'un poste standard ($2^{30} \times 16$ octets $\approx 17{,}2$ Go). Pourquoi ce coût dépend-t-il **seulement** de $n$ — et pas de la profondeur du circuit ni du nombre de portes ?
2. L'échantillonnage direct du §3 est quasi-constant par qubit (4 à 28 qubits, même ordre de temps) — et le carnet insiste : ce n'est **pas** Gottesman-Knill. Que mesure-t-il exactement, et où vit le vrai Gottesman-Knill ?
3. Le jeu Clifford+T est universel, mais un seul T ne rend pas chaque circuit difficile ; $\mathrm{BQP} \subseteq P$ reste ouvert. Qu'est-ce que le théorème de Gottesman-Knill établit précisément — et que ne dit-il **pas** sur le pouvoir du quantique ?

<details><summary>Réponses</summary>

1. Parce que l'état complet est stocké comme un vecteur de $2^n$ amplitudes, quelle que soit la séquence de portes qui l'a produit : chaque porte n'est qu'une matrice appliquée à ce vecteur. La mémoire paie la **description** de l'état, pas le circuit qui l'engendre — le tableau stabilisateur du 06b occupe lui aussi une mémoire dépendant de n, O(n²), et ses opérations dépendent des portes et mesures.
2. Il mesure le coût d'échantillonner une distribution **connue analytiquement** (uniforme : $n$ bits i.i.d., $O(n)$ par échantillon) — un point de comparaison de plancher, aucune table de stabilisateurs. La simulation par stabilisateurs — tableau de taille $O(n^2)$ et mise à jour des portes/mesures — est implémenté dans le carnet 06b.
3. Il établit que la sous-famille **Clifford** (H, S, CNOT) se simule en polynomial classique : cette terre-là ne demande pas de moteur quantique. Il ne dit rien des circuits universels : ajouter $T$ sort du seul jeu Clifford, sans prouver une difficulté exponentielle ; la méthode vectorielle générale coûte $O(2^n \cdot \mathrm{poly}(n, p))$. La question de fond — existe-t-il du quantique non simulable — reste exactement là.

</details>

Distinguez dans votre raisonnement décision déterministe et simulation probabiliste : la première hypothèse concerne P, la seconde BPP. Définir BQP inclut un temps polynomial et une erreur bornée sur les deux types d’instances, amplifiable par répétition.


## Pour aller plus loin

- **Carnet suivant** : [Complexity-06b — Aaronson-Gottesman : déquantification](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb) — l'arbitre par stabilisateurs qui exécute le cas Clifford en $O(n^2)$ et refuse poliment la première porte $T$.
- **Dans la même série** : [Complexity-06c — Classical at heart](Complexity-06c-ClassicalAtHeart-Python.ipynb) — la question miroir : implémenter une opération quantique à partir d'un oracle classique ; [Complexity-05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) — quand le mur du comptage devient un candidat d'avantage.
- **Carnet voisin d'une autre série** : [Lean-13b — La borne de Tsirelson](../SymbolicAI/Lean/Lean-13b-CHSH-Tsirelson-Native.ipynb) — la même frontière classique/quantique, cette fois digérée en Lean : le jeu de CHSH borne le score du classique à 2 et celui de l'intriqué à $2\sqrt{2}$.
- **Lectures** : les sources (Aaronson-Arkhipov 2013, Aaronson-Gottesman 2004, Nielsen-Chuang ch. 10, Aaronson 2009) figurent en fin de section « À retenir » ; la source Aaronson-Arkhipov (arXiv 1011.3245) est archivée au gisement partagé (dossier `Bibliographie IA/Consciousness/`), les trois autres références n'y ont pas de copie à ce jour.
